In [1]:
import pandas as pd
import xlrd
import openpyxl
from pandas import ExcelWriter
from datetime import datetime
import os
import numpy as np
#from arcgis.gis import GIS #DOESN'T Work on Mac
#from arcgis.features import SpatialDataFrame #DOESN'T Work on Mac
from shutil import copyfile
from openpyxl.utils.cell import get_column_letter
import urllib.request, json
import kobo_functions
from datetime import date

In [2]:
###USER DEFINED INPUTS BEGINNING
template_version = "20230720"
iso3 = "BGD"
validation_number = 1 #Please enter 1 or 2 ONLY
user_name = "Ibrahim Ali Ahmad" #Please write your name
country_questionnaire_file = r'BGD_Kobo_en_Questionnaire round 13_edited_7July.xlsx'

## Modfiy path to respiratory
template_questionnaire_location = r'/Users/ibrahimaliahmad/Library/Mobile Documents/com~apple~CloudDocs/Documents/FAO/Data in Emergencies (DIEM)/Working Files/Questionnaire Validation/Script/questionnaire_validation/Questionnaire Validation/Questionnaires/Questionnaire Templates'
country_questionnaires_location = r'/Users/ibrahimaliahmad/Library/Mobile Documents/com~apple~CloudDocs/Documents/FAO/Data in Emergencies (DIEM)/Working Files/Questionnaire Validation/Script/questionnaire_validation/Questionnaire Validation/Questionnaires/Questionnaire Country'
output_locations = r'/Users/ibrahimaliahmad/Library/Mobile Documents/com~apple~CloudDocs/Documents/FAO/Data in Emergencies (DIEM)/Working Files/Questionnaire Validation/Script/questionnaire_validation/Questionnaire Validation/Questionnaires/Validated Questionnaires/BGD/R13'
##

###USER DEFINED INPUTS END

In [3]:
enumerator = kobo_functions.detect_enumerator(country_questionnaire_file)
language = kobo_functions.detect_language(country_questionnaire_file)
date = date.today().strftime("%B %d, %Y")


from datetime import datetime

execution_messages = ''
execution_messages += "\n%s%s" % ("Country Questionnaire File: ", country_questionnaire_file)
execution_messages += "\n%s%s" % ("Country Questionnaire Type: ", enumerator)
execution_messages += "\n%s%s" % ("Country Questionnaire Language: ", language)
execution_messages += "\n%s%s" % ("The template used to validate the country questionnaire is: ", template_version)
execution_messages += "\n%s%s" % ("Validation number: ", validation_number)
execution_messages += "\n%s%s" % ("User: ", user_name)
execution_messages += "\n%s%s" % ("Validated on: ", date)

print(execution_messages)


country_questionnaire_file = os.path.join(country_questionnaires_location, country_questionnaire_file)
template_questionnaire_file = kobo_functions.detect_template(template_version,country_questionnaire_file)
template_questionnaire_file = os.path.join(template_questionnaire_location, template_questionnaire_file)


Country Questionnaire File: BGD_Kobo_en_Questionnaire round 13_edited_7July.xlsx
Country Questionnaire Type: kobo
Country Questionnaire Language: en
The template used to validate the country questionnaire is: 20230720
Validation number: 1
User: Ibrahim Ali Ahmad
Validated on: July 15, 2025


In [7]:
print("Reading TEMPLATE questionnaire and creating recap excel file")
questionnaire_file = template_questionnaire_file
temp_path = output_locations

startTime = datetime.now()
now = datetime.now().strftime('%Y%m%d%H%M%S')
coded_values_file = os.path.join(temp_path, "coded_values_%s_template.xlsx" % now) #intermediary output file with all categories and codes extracted from the questionnaire
writer = pd.ExcelWriter(coded_values_file, engine='xlsxwriter')
field_names_list = []
max_counter = 3000 #for testing purposes, we may need to limit the execution only to some items
#read_questionnaire(questionnaire_file, writer)

print("Reading COUNTRY questionnaire and creating recap excel file")
country_coded_values_file = os.path.join(temp_path, "coded_values_%s_country.xlsx" % now) #intermediary output file with all categories and codes extracted from the questionnaire
country_writer = pd.ExcelWriter(country_coded_values_file, engine='xlsxwriter')
field_names_list = []
max_counter = 3000 #for testing purposes, we may need to limit the execution only to some items
#read_questionnaire(country_questionnaire_file, country_writer)

Reading TEMPLATE questionnaire and creating recap excel file
Reading COUNTRY questionnaire and creating recap excel file


In [9]:
print("COUNTING NUMBER OF FIELDS\n")

T_qname_list_questions, T_qname_n_of_all_questions = kobo_functions.count_number_of_all_question_name(template_questionnaire_file)
C_qname_list_questions, C_qname_n_of_all_questions = kobo_functions.count_number_of_all_question_name(country_questionnaire_file)

message01 = "Number of fields in the Template Questionnaire: %s" % T_qname_n_of_all_questions
message02 = "Number of fields in the Country Questionnaire: %s" % C_qname_n_of_all_questions
print(message01)
print(message02)

if T_qname_n_of_all_questions != C_qname_n_of_all_questions:
    
    print ("\nDifferences detected are listed in the validation report")

    message03 = "\nAll Fields in country questionnaire and not in template questionnaire: %s" % list(set(C_qname_list_questions) - set(T_qname_list_questions))
    message04 = "All Fields in template questionnaire and not in country questionnaire: %s" % list(set(T_qname_list_questions) - set(C_qname_list_questions))
    execution_messages += "\n%s\n%s" % (message03, message04)
    #print (message03)
    #print (message04)
    
else:
    print("Country questionnaire contains all data fields")

COUNTING NUMBER OF FIELDS

Number of fields in the Template Questionnaire: 222
Number of fields in the Country Questionnaire: 303

Differences detected are listed in the validation report


In [11]:
print("Checking the conditionality of all questions")


questions_conditionality = os.path.join(temp_path, "conditionality_questionnaire_%s_%s_%s_%s.xlsx" % (enumerator,language,iso3, now)) 


conditionality_result_brief, conditionality_result_details = kobo_functions.check_all_questions(country_questionnaire_file, template_questionnaire_file, questions_conditionality)

print(conditionality_result_brief)
execution_messages += "\n\n" + conditionality_result_details



Checking the conditionality of all questions

DETECTING DIFFERENCES IN QUESTIONS: type

DETECTING DIFFERENCES IN QUESTIONS: label::English (en)

DETECTING DIFFERENCES IN QUESTIONS: hint::English (en)

DETECTING DIFFERENCES IN QUESTIONS: required

DETECTING DIFFERENCES IN QUESTIONS: appearance

DETECTING DIFFERENCES IN QUESTIONS: constraint

DETECTING DIFFERENCES IN QUESTIONS: choice_filter

DETECTING DIFFERENCES IN QUESTIONS: calculation

DETECTING DIFFERENCES IN QUESTIONS: Mandatory 



In [13]:
print ("CHECKING THAT ALL MANDATORY QUESTIONS ARE INCLUDED IN THE COUNTRY QUESTIONNAIRE")

quest_df = pd.read_excel(open(template_questionnaire_file, 'rb'), sheet_name='survey')
#print(list(quest_df.columns.values))
#quest_df.rename(columns={'Unnamed: 19': 'Mandatory'}, inplace=True)

list_mandatory_questions = quest_df.loc[((quest_df["name"].str.contains("hh_wealth_") == False ) & (quest_df["Mandatory "] == "yes"))]["name"].values.tolist() 
list_mandatory_questions = [s.strip() for s in list_mandatory_questions]
message5 = "\nFound %s mandatory questions in the template questionnaire" % len(list_mandatory_questions)
print(message5)

result =  all(elem in C_qname_list_questions  for elem in list_mandatory_questions)
if result:
    message6 = "Yes, country questionnaire contains all mandatory questions"   
else:
    message6 = "No, country questionnaire does not contains all mandatory questions.\nMissing question: %s " % list(set(list_mandatory_questions ) - set(C_qname_list_questions))
print(message6)
    
execution_messages += "\n%s\n%s" % (message5, message6)

CHECKING THAT ALL MANDATORY QUESTIONS ARE INCLUDED IN THE COUNTRY QUESTIONNAIRE

Found 117 mandatory questions in the template questionnaire
Yes, country questionnaire contains all mandatory questions


In [15]:
print ("CHECKING THAT RULES ON HH_WEALTH_ QUESTIONS ARE RESPECTED\n")

wealth_questions_in_country = [i for i in C_qname_list_questions if i.startswith('hh_wealth_')]
if len(wealth_questions_in_country) >= 1:
    message7 = "\n%s hh_wealth_ question found in the country questionnaire: RULE IS RESPECTED\n"% len(wealth_questions_in_country)

else:
    message7 = message7 = "\n%s hh_wealth_ questions found in the country questionnaire: it should be only one!\n" % len(wealth_questions_in_country)

print(message7)

execution_messages += "\n%s" % (message7)

CHECKING THAT RULES ON HH_WEALTH_ QUESTIONS ARE RESPECTED


1 hh_wealth_ question found in the country questionnaire: RULE IS RESPECTED



In [17]:
print ("CHECKING THAT RULES ON CS QUESTIONS ARE RESPECTED\n")

cs_questions_in_country = [i for i in C_qname_list_questions if i.startswith('cs')]
if len(cs_questions_in_country) > 0:
    cs_stress_questions_in_country = [i for i in C_qname_list_questions if i.startswith('cs_stress')]
    cs_emergency_questions_in_country = [i for i in C_qname_list_questions if i.startswith('cs_emergency')]
    cs_crisis_questions_in_country = [i for i in C_qname_list_questions if i.startswith('cs_crisis')]
    message8 = "There should be 10 or more CS questions in the Country Questionnaire: (4 stress + 3 crisis + 3 emergency)\n"
    
    
    if len(cs_questions_in_country) >= 10 and len(cs_stress_questions_in_country) >= 4 and len(cs_crisis_questions_in_country) >= 3 and len(cs_emergency_questions_in_country) >= 3:
        message8 += "\nRULE IS RESPECTED - "
    else :
        message8 += "\nMANUAL CHECK is needed - "
        
    message8 += "%s CS questions found in the country questionnaire" % len(cs_questions_in_country)
    message8 += "\nNumber of STRESS questions: %s" % len(cs_stress_questions_in_country)
    message8 += "\nNumber of CRISIS questions: %s" % len(cs_crisis_questions_in_country)
    message8 += "\nNumber of EMERGENCY questions: %s" % len(cs_emergency_questions_in_country)
else:
    message8 = "\nNo CS questions in the country questionnaire"

print(message8)

execution_messages += "\n%s" % (message8)

CHECKING THAT RULES ON CS QUESTIONS ARE RESPECTED

There should be 10 or more CS questions in the Country Questionnaire: (4 stress + 3 crisis + 3 emergency)

RULE IS RESPECTED - 16 CS questions found in the country questionnaire
Number of STRESS questions: 4
Number of CRISIS questions: 7
Number of EMERGENCY questions: 5


In [19]:
print ("CHECKING THAT RULES ON HDDS QUESTIONS ARE RESPECTED")

hdds_questions_in_country = [i for i in C_qname_list_questions if i.startswith('hdds_') ]

message9 = "\nThere should be more than 2 HDDS fields"

if len(hdds_questions_in_country) > 0:
    message9 += "\nNumber of HDDS questions: %s  - %s" % (len(hdds_questions_in_country), hdds_questions_in_country)
else:
    message9 += "\nNo HDDS questions in the country questionnaire"

execution_messages += "\n%s" % (message9)

print(message9)

CHECKING THAT RULES ON HDDS QUESTIONS ARE RESPECTED

There should be more than 2 HDDS fields
Number of HDDS questions: 2  - ['hdds_intro', 'hdds_confirmation']


In [21]:
print ("CHECKING THAT RULES ON CROP HARVEST QUESTIONS ARE RESPECTED")

crp_questions_in_country = [i for i in C_qname_list_questions if (i.startswith('crp_harv') and i != ('crp_harv_lastyr'))]

message09 = "\nThere should be either only \"crp_harv_change\" or 5 crp_harv questions from the template: \n\n"

if (len(crp_questions_in_country) >= 5) and ("crp_harv_change" in crp_questions_in_country):
    message09 += "RULE IS RESPECTED\nNumber of crp_harv questions: %s  - %s" % (len(crp_questions_in_country), crp_questions_in_country)
    
elif (len(crp_questions_in_country) == 1) and ("crp_harv_change" in crp_questions_in_country):
    message09 += "RULE IS RESPECTED\nNumber of crp_harv questions: %s  - %s" % (len(crp_questions_in_country), crp_questions_in_country)
else:
    message09 += "RULE IS NOT RESPECTED\nNumber of crp_harv questions: %s  - %s" % (len(crp_questions_in_country), crp_questions_in_country)

execution_messages += "\n%s" % (message09)

print(message09)

CHECKING THAT RULES ON CROP HARVEST QUESTIONS ARE RESPECTED

There should be either only "crp_harv_change" or 5 crp_harv questions from the template: 

RULE IS RESPECTED
Number of crp_harv questions: 5  - ['crp_harv_change', 'crp_harv_vol', 'crp_harv_unit', 'crp_harvunit', 'crp_harv_unit_kg']


In [23]:
print ("CREATE OUTPUT VALIDATED QUESTIONNAIRE FILE")
country_questionnaire_edited_file = os.path.join(temp_path, "validated_questionnaire_%s_%s_%s_%s.xlsx" % (enumerator,language,iso3, now)) 
copyfile(country_questionnaire_file, country_questionnaire_edited_file)
message =  "\n\nOutput validated country questionnaire: %s" % (country_questionnaire_edited_file)
print(message)
execution_messages += message


CREATE OUTPUT VALIDATED QUESTIONNAIRE FILE


Output validated country questionnaire: /Users/ibrahimaliahmad/Library/Mobile Documents/com~apple~CloudDocs/Documents/FAO/Data in Emergencies (DIEM)/Working Files/Questionnaire Validation/Script/questionnaire_validation/Questionnaire Validation/Questionnaires/Validated Questionnaires/BGD/R13/validated_questionnaire_kobo_en_BGD_20250715224134.xlsx


In [ ]:
print("EDITING THE SURVEY BY SORTING THE CROP LIST VALUES ACCORDING TO PRIORITISE")
n_of_choices = kobo_functions.sort_crop_list_by_selection(country_questionnaire_edited_file)
message13 = "\nNumber of most common crops selected in the Crop list sheet: %s" % n_of_choices
print(message13)
execution_messages += "\n\nSurvey edited by sorting the crop list according to the prioritise" + message13

print("Adding Crop2")
kobo_functions.sort_crop2_list_by_selection(country_questionnaire_edited_file)

print("Adding Crop3")
kobo_functions.sort_crop3_list_by_selection(country_questionnaire_edited_file)




In [25]:
### In some cases we will have to include Admin level 3 - will be refeleceted in email

print("INSERT ADMIN LIST WITH UPDATED BOUNDARIES")

sdf = kobo_functions.insert_adm_reference(country_questionnaire_edited_file,iso3)
#sdf.head()
execution_messages += "\n\nCreated Admin Info sheet with updated boundaries: %s" % (country_questionnaire_edited_file)

INSERT ADMIN LIST WITH UPDATED BOUNDARIES
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>


In [ ]:
print("EDITING THE SURVEY BY REPLACING VALUES ACCORDING ADDITIONAL INFO SHEET")

kobo_functions.update_question_label(country_questionnaire_edited_file)

kobo_functions.find_and_replace_strings_in_df(country_questionnaire_edited_file)
execution_messages += "\n\nSurvey edited by replacing values according the Additional Information sheet"

In [26]:
print("Create output file with execution messages")
text_file = os.path.join(temp_path, "questionnaire_validation_report_%s_%s.txt" % (iso3, now)) 
file = open(text_file, "w") 
file.write(execution_messages) 
file.close() 

Create output file with execution messages


In [27]:
print("INCLUDE THE FOLLOWING MESSAGE IN YOUR EMAIL\n")

print("Please ensure that the interim data is shared with the Hub team a couple of days after the commencement of data collection. This approach will enable the early detection and resolution of any issues related to data collection.")


INCLUDE THE FOLLOWING MESSAGE IN YOUR EMAIL

Please ensure that the interim data is shared with the Hub team a couple of days after the commencement of data collection. This approach will enable the early detection and resolution of any issues related to data collection.


In [28]:
print("Execution time: ", datetime.now() - startTime)

Execution time:  0:00:20.066629
